# Project A — Fetal Health Classification
EDA, preprocessing, training and evaluation of five scikit-learn classifiers (Logistic Regression, Decision Tree, KNN, Gaussian NB, SVM) on the Kaggle Fetal Health dataset.

- Dataset: https://www.kaggle.com/datasets/andrewmvd/fetal-health-classification (not included in this repository)
- Run in the BITS Pilani Virtual Lab: Python 3.14, scikit-learn 1.8.0
- Outputs saved to `models/`: five `.joblib` pipelines, `metadata.json`, `test_set.csv`, `metrics_project_a.csv`


In [2]:
# =========================================================
# PROJECT A — Fetal Health Classification: EDA + TRAINING (single cell)
# Fixes applied: (1) exact-name leakage check, (2) conflict
# count labelled as groups, (3) display() for mid-cell tables
# =========================================================

# ---------------------------------------------------------
# SETUP & IMPORTS
# ---------------------------------------------------------
import pandas as pd
from pathlib import Path
from IPython.display import display

# Show all columns when printing dataframes
pd.set_option("display.max_columns", None)

# Set the path to the folder containing the dataset
data_dir = Path("/drive/notebooks")
csv_path = data_dir / "fetal_health.csv"

# Quick check of what files are in the directory
print("Files in directory:")
for f in sorted(data_dir.iterdir()):
    print(f" - {f.name} ({f.stat().st_size / 1024:.1f} KB)")


# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
df = pd.read_csv(csv_path)
print(f"\nDataset shape: {df.shape[0]} rows, {df.shape[1]} columns")

target = "fetal_health"
features = [col for col in df.columns if col != target]

display(df.head())


# ---------------------------------------------------------
# DATA TYPES & MISSING VALUES
# ---------------------------------------------------------
df.info()

print("\n--- Missing Values ---")
missing = df.isna().sum()
if missing.any():
    print(missing[missing > 0])
else:
    print("No missing values found.")

print("\n--- Duplicates ---")
print(f"Exact duplicate rows: {df.duplicated().sum()}")

# Groups of rows with identical features but different target labels
conflicts = df[df.duplicated(subset=features, keep=False)].groupby(features)[target].nunique()
print(f"Groups of identical features with different labels: {conflicts[conflicts > 1].count()}")


# ---------------------------------------------------------
# TARGET DISTRIBUTION
# ---------------------------------------------------------
print("\n--- Target Variable Breakdown ---")
class_counts = df[target].value_counts().sort_index()
class_pct = (df[target].value_counts(normalize=True).sort_index() * 100).round(2)

dist_df = pd.DataFrame({"Count": class_counts, "Percentage (%)": class_pct})
print(dist_df)

imbalance = class_counts.max() / class_counts.min()
print(f"\nImbalance ratio (majority to minority): {imbalance:.2f}x")


# ---------------------------------------------------------
# FEATURE STATISTICS
# ---------------------------------------------------------
# Summary of the numerical features, including skewness and unique counts
stats = df[features].describe().T
stats["skewness"] = df[features].skew().round(2)
stats["unique_vals"] = df[features].nunique()

display(stats.round(3))


# ---------------------------------------------------------
# CARDINALITY CHECKS
# ---------------------------------------------------------
num_cols = df[features].select_dtypes(include="number").columns.tolist()
cat_cols = df[features].select_dtypes(exclude="number").columns.tolist()

print(f"Numeric columns ({len(num_cols)}): {num_cols}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

# Flag numeric columns that might actually be categorical (very few unique values)
low_cardinality = [col for col in num_cols if df[col].nunique() <= 10]

if low_cardinality:
    print("\nNumeric columns with <= 10 distinct values (worth inspecting):")
    for col in low_cardinality:
        print(f"  {col}: {sorted(df[col].unique().tolist())}")


# ---------------------------------------------------------
# DATA LEAKAGE & CORRELATION
# ---------------------------------------------------------
# Exact-name match, so columns like "histogram_width" are not falsely flagged
suspicious_keywords = {"class", "nsp", "id"}
suspicious_cols = [c for c in df.columns if c.lower() in suspicious_keywords]

print("\n--- Leakage Check ---")
print(f"Suspicious columns (IDs or label copies): {suspicious_cols if suspicious_cols else 'None'}")

# Spearman = Pearson correlation on ranks (no SciPy needed)
correlations = df[features].rank().corrwith(df[target].rank()).abs().sort_values(ascending=False)

print("\nTop 8 features correlated with the target (Spearman Rank):")
print(correlations.head(8).round(3))


print("\n" + "=" * 70 + "\nSTAGE 3 — PREPROCESSING, TRAINING, EVALUATION\n" + "=" * 70)


# ---------------------------------------------------------
# P0 — IMPORTS, VERSIONS, PATHS
# ---------------------------------------------------------
print('\n--- P0 ---')
import sys, json, time
import numpy as np, pandas as pd, sklearn, joblib
from pathlib import Path
from IPython.display import display

print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__, "| joblib", joblib.__version__)
RANDOM_STATE = 42
DATA_PATH = csv_path                      # reuses the path from the EDA cell
MODEL_DIR = Path("models"); MODEL_DIR.mkdir(exist_ok=True)


# ---------------------------------------------------------
# P1 — CLEAN: int target, drop 13 duplicates, drop histogram_width
# ---------------------------------------------------------
print('\n--- P1 ---')
df = pd.read_csv(DATA_PATH)
TARGET = "fetal_health"
CLASS_NAMES = {1: "Normal", 2: "Suspect", 3: "Pathological"}
print("Raw shape:", df.shape, "| missing values:", int(df.isna().sum().sum()))

df[TARGET] = df[TARGET].astype(int)                 # 1.0 -> 1, etc.
df = df.drop_duplicates().reset_index(drop=True)     # remove 13 exact copies
df = df.drop(columns=["histogram_width"])            # exact copy of max - min
print("Clean shape:", df.shape)
print(df[TARGET].value_counts().sort_index().rename(index=CLASS_NAMES))


# ---------------------------------------------------------
# P2 — STRATIFIED 80/20 TRAIN/TEST SPLIT
# ---------------------------------------------------------
print('\n--- P2 ---')
from sklearn.model_selection import train_test_split

X = df.drop(columns=[TARGET]); y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)
split = pd.DataFrame({"train": y_train.value_counts().sort_index(),
                      "test": y_test.value_counts().sort_index()})
split["train %"] = (split["train"] / split["train"].sum() * 100).round(2)
split["test %"] = (split["test"] / split["test"].sum() * 100).round(2)
display(split.rename(index=CLASS_NAMES))


# ---------------------------------------------------------
# P3 — DEFINE THE FIVE PIPELINES
# ---------------------------------------------------------
print('\n--- P3 ---')
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE))]),
    "Decision Tree": Pipeline([("clf", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE))]),
    "KNN": Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=5))]),
    "Gaussian NB": Pipeline([("power", PowerTransformer(method="yeo-johnson", standardize=True)), ("clf", GaussianNB())]),
    "SVM": Pipeline([("scaler", StandardScaler()),
        ("clf", CalibratedClassifierCV(SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE),
                                       method="sigmoid", cv=5, ensemble=False))]),
}
for name, pipe in models.items():
    print(f"{name:20s} ->", " -> ".join(pipe.named_steps))


# ---------------------------------------------------------
# P4 — TRAIN ON TRAINING SET ONLY
# ---------------------------------------------------------
print('\n--- P4 ---')
for name, pipe in models.items():
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    print(f"{name:20s} trained in {time.perf_counter() - t0:.2f} s | classes: {[int(c) for c in pipe.classes_]}")


# ---------------------------------------------------------
# P5 — EVALUATE ON THE SAME TEST SET (macro averaging, OvR AUC)
# ---------------------------------------------------------
print('\n--- P5 ---')
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

def evaluate(pipe, X_te, y_te):
    y_pred = pipe.predict(X_te)
    y_proba = pipe.predict_proba(X_te)
    return {"Accuracy": accuracy_score(y_te, y_pred),
            "AUC (OvR, macro)": roc_auc_score(y_te, y_proba, multi_class="ovr",
                                              average="macro", labels=pipe.classes_),
            "Precision (macro)": precision_score(y_te, y_pred, average="macro", zero_division=0),
            "Recall (macro)": recall_score(y_te, y_pred, average="macro", zero_division=0),
            "F1 (macro)": f1_score(y_te, y_pred, average="macro", zero_division=0),
            "MCC": matthews_corrcoef(y_te, y_pred)}

results = pd.DataFrame({name: evaluate(pipe, X_test, y_test) for name, pipe in models.items()}).T
display(results.round(4))


# ---------------------------------------------------------
# P6 — SAVE MODELS, METADATA, TEST SET, METRICS
# ---------------------------------------------------------
print('\n--- P6 ---')
for name, pipe in models.items():
    fname = MODEL_DIR / (name.lower().replace(" ", "_") + ".joblib")
    joblib.dump(pipe, fname)
    print(f"Saved {fname} ({fname.stat().st_size / 1024:.1f} KB)")

meta = {"target": TARGET, "class_names": CLASS_NAMES, "feature_columns": list(X.columns),
        "random_state": RANDOM_STATE, "test_size": 0.2, "sklearn_version": sklearn.__version__,
        "models": {n: n.lower().replace(" ", "_") + ".joblib" for n in models}}
(MODEL_DIR / "metadata.json").write_text(json.dumps(meta, indent=2))
X_test.assign(**{TARGET: y_test}).to_csv(MODEL_DIR / "test_set.csv", index=False)
results.to_csv(MODEL_DIR / "metrics_project_a.csv")
print("Saved metadata.json, test_set.csv, metrics_project_a.csv")


# ---------------------------------------------------------
# P7 — RELOAD SAVED MODELS AND VERIFY
# ---------------------------------------------------------
print('\n--- P7 ---')
for name, pipe in models.items():
    loaded = joblib.load(MODEL_DIR / (name.lower().replace(" ", "_") + ".joblib"))
    same = (loaded.predict(X_test) == pipe.predict(X_test)).all()
    print(f"{name:20s} reload OK, identical predictions: {same}")

Files in directory:
 - Intro.ipynb (29.7 KB)
 - Lorenz.ipynb (6.4 KB)
 - Mandelbrot.ipynb (7.1 KB)
 - Untitled Folder (0.0 KB)
 - cpp-third-party-libs.ipynb (6.8 KB)
 - cpp-tiny-ray-tracer.ipynb (13.3 KB)
 - cpp.ipynb (15.2 KB)
 - fetal_health.csv (223.4 KB)
 - r.ipynb (8.6 KB)
 - sqlite.ipynb (6.9 KB)

Dataset shape: 2126 rows, 22 columns


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,mean_value_of_long_term_variability,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,2.4,64.0,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,10.4,130.0,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,13.4,130.0,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,23.0,117.0,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,19.9,117.0,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


<class 'pandas.DataFrame'>
RangeIndex: 2126 entries, 0 to 2125
Data columns (total 22 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   baseline value                                          2126 non-null   float64
 1   accelerations                                           2126 non-null   float64
 2   fetal_movement                                          2126 non-null   float64
 3   uterine_contractions                                    2126 non-null   float64
 4   light_decelerations                                     2126 non-null   float64
 5   severe_decelerations                                    2126 non-null   float64
 6   prolongued_decelerations                                2126 non-null   float64
 7   abnormal_short_term_variability                         2126 non-null   float64
 8   mean_value_of_short_term_variability             

,count,mean,std,min,25%,50%,75%,max,skewness,unique_vals
baseline value,2126.0,133.304,9.841,106.0,126.000,133.000,140.000,160.000,0.02,48
accelerations,2126.0,0.003,0.004,0.0,0.000,0.002,0.006,0.019,1.20,20
fetal_movement,2126.0,0.009,0.047,0.0,0.000,0.000,0.003,0.481,7.81,102
uterine_contractions,2126.0,0.004,0.003,0.0,0.002,0.004,0.007,0.015,0.16,16
light_decelerations,2126.0,0.002,0.003,0.0,0.000,0.000,0.003,0.015,1.72,16
severe_decelerations,2126.0,0.000,0.000,0.0,0.000,0.000,0.000,0.001,17.35,2
prolongued_decelerations,2126.0,0.000,0.001,0.0,0.000,0.000,0.000,0.005,4.32,6
abnormal_short_term_variability,2126.0,46.990,17.193,12.0,32.000,49.000,61.000,87.000,-0.01,75
mean_value_of_short_term_variability,2126.0,1.333,0.883,0.2,0.700,1.200,1.700,7.000,1.66,57
percentage_of_time_with_abnormal_long_term_variability,2126.0,9.847,18.397,0.0,0.000,0.000,11.000,91.000,2.20,87


Numeric columns (21): ['baseline value', 'accelerations', 'fetal_movement', 'uterine_contractions', 'light_decelerations', 'severe_decelerations', 'prolongued_decelerations', 'abnormal_short_term_variability', 'mean_value_of_short_term_variability', 'percentage_of_time_with_abnormal_long_term_variability', 'mean_value_of_long_term_variability', 'histogram_width', 'histogram_min', 'histogram_max', 'histogram_number_of_peaks', 'histogram_number_of_zeroes', 'histogram_mode', 'histogram_mean', 'histogram_median', 'histogram_variance', 'histogram_tendency']
Categorical columns (0): []

Numeric columns with <= 10 distinct values (worth inspecting):
  severe_decelerations: [0.0, 0.001]
  prolongued_decelerations: [0.0, 0.001, 0.002, 0.003, 0.004, 0.005]
  histogram_number_of_zeroes: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 7.0, 8.0, 10.0]
  histogram_tendency: [-1.0, 0.0, 1.0]

--- Leakage Check ---
Suspicious columns (IDs or label copies): None

Top 8 features correlated with the target (Spearman Rank

,train,test,train %,test %
fetal_health,,,,
Normal,1316,330,77.87,78.01
Suspect,234,58,13.85,13.71
Pathological,140,35,8.28,8.27



--- P3 ---
Logistic Regression  -> scaler -> clf
Decision Tree        -> clf
KNN                  -> scaler -> clf
Gaussian NB          -> power -> clf
SVM                  -> scaler -> clf

--- P4 ---
Logistic Regression  trained in 0.07 s | classes: [1, 2, 3]
Decision Tree        trained in 0.02 s | classes: [1, 2, 3]
KNN                  trained in 0.00 s | classes: [1, 2, 3]
Gaussian NB          trained in 0.51 s | classes: [1, 2, 3]
SVM                  trained in 0.51 s | classes: [1, 2, 3]

--- P5 ---


/lib/python3.14/site-packages/threadpoolctl.py:1135: RuntimeWarning: JsProxy.as_object_map() is deprecated. Use as_py_json() instead.
  for filepath in LDSO.loadedLibsByName.as_object_map():


,Accuracy,"AUC (OvR, macro)",Precision (macro),Recall (macro),F1 (macro),MCC
Logistic Regression,0.8676,0.9637,0.7526,0.8402,0.7869,0.6914
Decision Tree,0.9314,0.8938,0.9042,0.8646,0.8820,0.8065
KNN,0.9007,0.9527,0.8654,0.7502,0.7981,0.7089
Gaussian NB,0.7470,0.8503,0.6540,0.7273,0.6461,0.5356
SVM,0.8983,0.9604,0.8109,0.8268,0.8177,0.7226



--- P6 ---
Saved models/logistic_regression.joblib (3.0 KB)
Saved models/decision_tree.joblib (25.2 KB)
Saved models/knn.joblib (273.1 KB)
Saved models/gaussian_nb.joblib (3.7 KB)
Saved models/svm.joblib (119.4 KB)
Saved metadata.json, test_set.csv, metrics_project_a.csv

--- P7 ---
Logistic Regression  reload OK, identical predictions: True
Decision Tree        reload OK, identical predictions: True
KNN                  reload OK, identical predictions: True
Gaussian NB          reload OK, identical predictions: True
SVM                  reload OK, identical predictions: True
